# GP Model: Input → Output Mapping & Posterior Sampling

## Campaign Design

Each campaign was a Bayesian Optimization loop targeting a **different QW phase**:

| Campaign | BO Target | R_BAAc range | Temperature | Strategy |
|----------|-----------|-------------|-------------|----------|
| **N1**   | Maximize **QW 1** | 0.68 – 2.50 | 60, 105, 150°C | Broad exploration, high BA content |
| **N2**   | Maximize **QW 2** | 0.40 – 1.52 | 60 – 150°C (8 levels) | Medium focus, finer T resolution |
| **N3**   | Maximize **QW 3** | 0.40 – 1.10 | 60 – 150°C (6 levels) | Narrow, converging to lower BA |

The campaigns progressively **narrowed** the input space as the BO converged: N1 spread wide to discover that high BA ratios favour n=1, then N2/N3 zoomed in on the lower-BA region to push toward higher n-values.

**This notebook:**
1. Visualises campaign coverage and QW distributions
2. Fits a **Gaussian Process** for each QW output: `inputs (4D) → QW fraction`
3. **Samples from the GP posterior** to generate a dense synthetic dataset
4. Exports the samples for use in the Grid World simulation

In [ ]:
import io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Libraries loaded.')

## 1. Load & Deduplicate All Campaign Data

In [ ]:
BASE = '.'

with open(f'{BASE}/N1_Campaign/Data_Tracked/N1Campaign_data.txt') as f:
    df1 = pd.read_csv(io.StringIO(f.read()))
df1['campaign'] = 'N1'

df2 = pd.read_csv(f'{BASE}/N2_Campaign/Data_Tracked/N2Campaign_data.csv')
df2['campaign'] = 'N2'

df3 = pd.read_csv(f'{BASE}/N3_Campaign/Data_Tracked/N3Campaign_data.csv')
df3['campaign'] = 'N3'

# Deduplicate: some rows are shared across campaigns (same measurement file)
# Keep the first occurrence of each unique 'files' entry
df_all = pd.concat([df1, df2, df3], ignore_index=True)
df_all = df_all.drop_duplicates(subset='files', keep='first').reset_index(drop=True)

QW_COLS    = [f'QW {i}' for i in range(1, 13)] + ['QW 99']
INPUT_COLS = ['Anneal Time', 'Temperature', 'R BAAc', 'R MAI']
CAMPAIGN_TARGETS = {'N1': 'QW 1', 'N2': 'QW 2', 'N3': 'QW 3'}

df_all = df_all.dropna(subset=INPUT_COLS + QW_COLS).reset_index(drop=True)

print(f'Total unique data points: {len(df_all)}')
print(f'Campaign origin: {df_all.campaign.value_counts().to_dict()}')
print()
print('Input ranges:')
print(df_all[INPUT_COLS].describe().round(3))

## 2. Campaign Coverage & QW Distribution Overview

In [ ]:
camp_colors  = {'N1': '#e74c3c', 'N2': '#f39c12', 'N3': '#2980b9'}
camp_markers = {'N1': 'o', 'N2': 's', 'N3': '^'}

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# --- Top row: input space coverage ---
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[0, 2])

for camp, grp in df_all.groupby('campaign'):
    kw = dict(c=camp_colors[camp], marker=camp_markers[camp],
              s=40, alpha=0.7, edgecolors='k', linewidths=0.3, label=camp)
    ax1.scatter(grp['R BAAc'], grp['Temperature'], **kw)
    ax2.scatter(grp['R BAAc'], grp['R MAI'],       **kw)
    ax3.scatter(grp['Anneal Time'], grp['Temperature'], **kw)

ax1.set_xlabel('R_BAAc'); ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Input Coverage:\nR_BAAc vs Temperature')
ax1.legend(fontsize=8)

ax2.set_xlabel('R_BAAc'); ax2.set_ylabel('R_MAI')
ax2.set_title('Input Coverage:\nR_BAAc vs R_MAI')

ax3.set_xlabel('Anneal Time (s)'); ax3.set_ylabel('Temperature (°C)')
ax3.set_title('Input Coverage:\nAnneal Time vs Temperature')

# --- Bottom row: mean QW distribution per campaign ---
ax4 = fig.add_subplot(gs[1, :])
x   = np.arange(len(QW_COLS))
w   = 0.25

for i, (camp, grp) in enumerate(df_all.groupby('campaign')):
    means = grp[QW_COLS].mean().values
    bars  = ax4.bar(x + (i - 1) * w, means, w,
                    color=camp_colors[camp], alpha=0.85,
                    edgecolor='black', linewidth=0.4,
                    label=f'{camp} (target: {CAMPAIGN_TARGETS[camp]})')

# Highlight each campaign's target QW
for i, camp in enumerate(['N1', 'N2', 'N3']):
    tgt_idx = QW_COLS.index(CAMPAIGN_TARGETS[camp])
    ax4.axvline(tgt_idx + (i - 1) * w, color=camp_colors[camp],
                linestyle='--', linewidth=1.5, alpha=0.6)

ax4.set_xticks(x)
ax4.set_xticklabels(QW_COLS, rotation=45, ha='right', fontsize=9)
ax4.set_ylabel('Mean Fraction')
ax4.set_title('Mean QW Distribution per Campaign  (dashed lines = BO target for each campaign)')
ax4.legend(fontsize=9)

plt.savefig('campaign_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved campaign_overview.png')

## 3. Fit Gaussian Process Models

We fit **one GP per QW output** (separate independent GPs).

**Kernel:** `Constant × Matérn(ν=2.5) + WhiteKernel`
- Matérn(ν=2.5) is the standard choice for physical systems — once-differentiable, captures smooth but non-trivial surfaces
- WhiteKernel models measurement noise
- Inputs are standardised (zero mean, unit variance) before fitting

The GP gives us:
- `μ(x*)` — predicted mean QW fraction at any new condition `x*`
- `σ(x*)` — uncertainty (high where data is sparse)

In [ ]:
X_raw = df_all[INPUT_COLS].values   # (N, 4)
Y_raw = df_all[QW_COLS].values      # (N, 13)

# Standardise inputs
scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

def make_kernel():
    return ConstantKernel(1.0, (0.1, 10)) * Matern(length_scale=1.0, nu=2.5) \
           + WhiteKernel(noise_level=0.01, noise_level_bounds=(1e-4, 0.5))

# Fit one GP per QW output
gps = {}
for j, qw in enumerate(QW_COLS):
    y = Y_raw[:, j]
    gp = GaussianProcessRegressor(
        kernel=make_kernel(),
        n_restarts_optimizer=5,
        normalize_y=True,
        alpha=1e-6,
    )
    gp.fit(X, y)
    gps[qw] = gp
    print(f'  {qw}: fitted  log-marginal-likelihood = {gp.log_marginal_likelihood_value_:.2f}')

print('\nAll GPs fitted.')

## 4. GP Mean & Uncertainty: 2D Slices (T vs R_BAAc)

We slice the 4D space at **median `Anneal Time`** and **median `R_MAI`** to visualise the two most important axes.

In [ ]:
# Fixed values for the 2D slice
ANNEAL_FIXED = float(np.median(df_all['Anneal Time']))
RMAI_FIXED   = float(np.median(df_all['R MAI']))

N_GRID    = 40
baac_vals = np.linspace(df_all['R BAAc'].min(), df_all['R BAAc'].max(), N_GRID)
temp_vals = np.linspace(df_all['Temperature'].min(), df_all['Temperature'].max(), N_GRID)
BB, TT    = np.meshgrid(baac_vals, temp_vals)

# Build query grid (N_GRID^2, 4)
X_query_raw = np.column_stack([
    np.full(N_GRID**2, ANNEAL_FIXED),
    TT.ravel(),
    BB.ravel(),
    np.full(N_GRID**2, RMAI_FIXED),
])
X_query = scaler.transform(X_query_raw)

print(f'2D slice at Anneal={ANNEAL_FIXED:.0f}s, R_MAI={RMAI_FIXED:.3f}')
print(f'Grid: {N_GRID}x{N_GRID} = {N_GRID**2} query points')

In [ ]:
# Plot GP mean and std for the three campaign target QWs
TARGET_QWS = ['QW 1', 'QW 2', 'QW 3']

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for col, qw in enumerate(TARGET_QWS):
    mu, sigma = gps[qw].predict(X_query, return_std=True)
    mu    = mu.reshape(N_GRID, N_GRID)
    sigma = sigma.reshape(N_GRID, N_GRID)

    ext = [baac_vals.min(), baac_vals.max(), temp_vals.min(), temp_vals.max()]

    # Mean surface
    ax_mu = axes[0, col]
    im1   = ax_mu.imshow(mu, origin='lower', aspect='auto', extent=ext, cmap='viridis')
    plt.colorbar(im1, ax=ax_mu, label='Predicted fraction')

    # Overlay actual data for this campaign's target
    camp  = {'QW 1':'N1','QW 2':'N2','QW 3':'N3'}[qw]
    grp   = df_all[df_all.campaign == camp]
    sc    = ax_mu.scatter(grp['R BAAc'], grp['Temperature'],
                          c=grp[qw], cmap='viridis', s=30, edgecolors='white',
                          linewidths=0.5, vmin=mu.min(), vmax=mu.max(), zorder=5)
    ax_mu.set_title(f'GP Mean: {qw}\n(data points from {camp} campaign)')
    ax_mu.set_xlabel('R_BAAc'); ax_mu.set_ylabel('Temperature (°C)')

    # Uncertainty surface
    ax_std = axes[1, col]
    im2    = ax_std.imshow(sigma, origin='lower', aspect='auto', extent=ext, cmap='plasma')
    plt.colorbar(im2, ax=ax_std, label='Std dev (uncertainty)')

    # Show ALL data points (not just this campaign) to see where uncertainty is low
    ax_std.scatter(df_all['R BAAc'], df_all['Temperature'],
                   c='white', s=10, alpha=0.4, zorder=5)
    ax_std.set_title(f'GP Uncertainty: {qw}\n(white dots = any measured point)')
    ax_std.set_xlabel('R_BAAc'); ax_std.set_ylabel('Temperature (°C)')

plt.suptitle(f'GP Predictions at Anneal={ANNEAL_FIXED:.0f}s, R_MAI={RMAI_FIXED:.2f}',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('gp_mean_uncertainty.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved gp_mean_uncertainty.png')

## 5. Sample from the GP Posterior

Instead of only using the GP mean, we **draw samples from the full posterior distribution**:
$$
\tilde{f}(x) \sim \mathcal{GP}(\mu(x),\, k(x,x'))
$$
This gives us a **synthetic dataset** that:
- Honours what was actually measured (low variance near data points)
- Explores plausible outcomes in unmeasured regions (high variance = wider spread of samples)
- Can populate a much denser grid world than the 456 real experiments alone allow

In [ ]:
# ---------------------------------------------------------------
# CONFIGURE: how to build the synthetic grid
# ---------------------------------------------------------------
N_SYNTH_GRID = 25     # dense grid per axis (25x25 = 625 input combinations)
N_SAMPLES    = 5      # GP posterior samples per grid point
# Fixed Anneal Time and R_MAI (can also loop over these)
ANNEAL_SYNTH = 20.0   # seconds
RMAI_SYNTH   = 1.45   # median value

# Build the 2D synthesis grid
b_syn = np.linspace(df_all['R BAAc'].min(), df_all['R BAAc'].max(), N_SYNTH_GRID)
t_syn = np.linspace(df_all['Temperature'].min(), df_all['Temperature'].max(), N_SYNTH_GRID)
Bsyn, Tsyn = np.meshgrid(b_syn, t_syn)

X_syn_raw = np.column_stack([
    np.full(N_SYNTH_GRID**2, ANNEAL_SYNTH),
    Tsyn.ravel(),
    Bsyn.ravel(),
    np.full(N_SYNTH_GRID**2, RMAI_SYNTH),
])
X_syn = scaler.transform(X_syn_raw)

# Draw samples for each QW from the posterior
synth_samples = {}  # qw -> array of shape (N_GRID^2 * N_SAMPLES, )
synth_mean    = {}  # qw -> mean prediction, shape (N_GRID^2,)
synth_std     = {}

for qw in QW_COLS:
    mu_s, std_s   = gps[qw].predict(X_syn, return_std=True)
    # Draw N_SAMPLES independent posterior sample functions
    post_samples  = gps[qw].sample_y(X_syn, n_samples=N_SAMPLES, random_state=42)
    # post_samples shape: (N_GRID^2, N_SAMPLES)
    synth_samples[qw] = post_samples
    synth_mean[qw]    = mu_s
    synth_std[qw]     = std_s

# Build a flat synthetic dataframe: repeat each grid point N_SAMPLES times
rows = []
n_pts = N_SYNTH_GRID ** 2
for s in range(N_SAMPLES):
    block = pd.DataFrame(X_syn_raw, columns=INPUT_COLS)
    for qw in QW_COLS:
        block[qw] = synth_samples[qw][:, s]
    block['sample_id'] = s
    rows.append(block)

df_syn = pd.concat(rows, ignore_index=True)
df_syn['source'] = 'GP_sample'

# Clip to [0, 1] — QW fractions must be non-negative
df_syn[QW_COLS] = df_syn[QW_COLS].clip(lower=0.0, upper=1.0)

print(f'Synthetic dataset: {len(df_syn)} rows '
      f'({N_SYNTH_GRID}x{N_SYNTH_GRID} grid x {N_SAMPLES} samples)')
print(df_syn[QW_COLS[:4]].describe().round(3))

## 6. Visualise Posterior Samples vs Real Data

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for col, qw in enumerate(TARGET_QWS):
    ax  = axes[col]
    ext = [b_syn.min(), b_syn.max(), t_syn.min(), t_syn.max()]

    # GP mean as background heatmap
    mean_map = synth_mean[qw].reshape(N_SYNTH_GRID, N_SYNTH_GRID)
    im = ax.imshow(mean_map, origin='lower', aspect='auto',
                   extent=ext, cmap='viridis', alpha=0.85)
    plt.colorbar(im, ax=ax, label=f'{qw} fraction')

    # Posterior sample scatter (one colour per sample)
    cmap_s = plt.cm.get_cmap('cool', N_SAMPLES)
    for s in range(N_SAMPLES):
        block = df_syn[df_syn.sample_id == s]
        ax.scatter(block['R BAAc'], block['Temperature'],
                   c=[cmap_s(s)]*len(block), s=4, alpha=0.3, zorder=4)

    # Real data overlay
    camp = {'QW 1':'N1','QW 2':'N2','QW 3':'N3'}[qw]
    real = df_all[df_all.campaign == camp]
    ax.scatter(real['R BAAc'], real['Temperature'],
               c=real[qw], cmap='viridis', s=60, edgecolors='red',
               linewidths=1.2, zorder=10, label=f'{camp} real data')

    ax.set_xlabel('R_BAAc'); ax.set_ylabel('Temperature (°C)')
    ax.set_title(f'{qw}  |  GP mean + {N_SAMPLES} posterior samples\n(red border = real measurements)')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('gp_posterior_samples.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved gp_posterior_samples.png')

## 7. GP Mean Reward Maps for All 3 Campaign Targets

Using the GP mean to predict a dense 20×20 reward grid for each BO objective — this is a much smoother and denser version of what we built in `GridWorld_Simulation.ipynb`.

In [ ]:
GW_SIZE = 20
b_gw    = np.linspace(df_all['R BAAc'].min(), df_all['R BAAc'].max(), GW_SIZE)
t_gw    = np.linspace(df_all['Temperature'].min(), df_all['Temperature'].max(), GW_SIZE)
Bgw, Tgw = np.meshgrid(b_gw, t_gw)

X_gw_raw = np.column_stack([
    np.full(GW_SIZE**2, ANNEAL_SYNTH),
    Tgw.ravel(), Bgw.ravel(),
    np.full(GW_SIZE**2, RMAI_SYNTH),
])
X_gw = scaler.transform(X_gw_raw)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
tick_idx  = [0, 4, 9, 14, 19]

reward_maps_gp = {}   # save for GridWorld use

for col, qw in enumerate(TARGET_QWS):
    mu_gw, std_gw = gps[qw].predict(X_gw, return_std=True)
    reward_map_gp = mu_gw.reshape(GW_SIZE, GW_SIZE)
    reward_maps_gp[qw] = reward_map_gp

    ax  = axes[col]
    im  = ax.imshow(reward_map_gp, origin='lower', cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax, label=f'{qw} fraction')

    # Uncertainty contours
    std_map = std_gw.reshape(GW_SIZE, GW_SIZE)
    ax.contour(std_map, levels=4, colors='white', alpha=0.4, linewidths=0.7,
               origin='lower')

    best_r, best_c = np.unravel_index(reward_map_gp.argmax(), reward_map_gp.shape)
    ax.plot(best_c, best_r, 'w*', markersize=16, markeredgecolor='black',
            markeredgewidth=1, zorder=10)

    ax.set_xticks(tick_idx)
    ax.set_xticklabels([f'{b_gw[i]:.2f}' for i in tick_idx], fontsize=7)
    ax.set_yticks(tick_idx)
    ax.set_yticklabels([f'{t_gw[i]:.0f}°C' for i in tick_idx], fontsize=7)
    ax.set_xlabel('R_BAAc'); ax.set_ylabel('Temperature')

    camp   = {'QW 1':'N1','QW 2':'N2','QW 3':'N3'}[qw]
    br_val = reward_map_gp[best_r, best_c]
    ax.set_title(f'GP Mean: {qw} (target of {camp})\n'
                 f'★ best: T={t_gw[best_r]:.0f}°C, R_BAAc={b_gw[best_c]:.2f} ({br_val:.3f})')

plt.suptitle(f'GP-predicted {GW_SIZE}x{GW_SIZE} Grid World at Anneal={ANNEAL_SYNTH}s, R_MAI={RMAI_SYNTH}\n'
             'White contours = GP uncertainty; ★ = optimal cell per GP', y=1.02)
plt.tight_layout()
plt.savefig('gp_gridworld_maps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved gp_gridworld_maps.png')

## 8. GP-Populated Dense Dataset

Combine **real measurements + GP posterior samples** into one dataset — this is what you'd feed into a denser Grid World or a new BO campaign.

Key idea: in unexplored regions the GP samples spread out (high variance), giving the grid world agent a realistic representation of uncertainty — it can discover that some cells are risky/unknown.

In [ ]:
# Compare uncertainty at real data points vs synthetic points
X_real_scaled = scaler.transform(df_all[INPUT_COLS].values)

fig, axes = plt.subplots(1, 3, figsize=(17, 4))

for col, qw in enumerate(TARGET_QWS):
    _, std_real = gps[qw].predict(X_real_scaled, return_std=True)
    _, std_syn  = gps[qw].predict(X_syn, return_std=True)

    ax = axes[col]
    ax.hist(std_real, bins=20, alpha=0.7, color='steelblue', label=f'Real data (n={len(std_real)})')
    ax.hist(std_syn,  bins=20, alpha=0.7, color='tomato',    label=f'GP samples (n={len(std_syn)})')
    ax.set_xlabel('GP uncertainty (std dev)')
    ax.set_ylabel('Count')
    ax.set_title(f'Uncertainty Distribution: {qw}')
    ax.legend(fontsize=8)

plt.suptitle('Real data has low uncertainty (near observations);\n'
             'GP samples include high-uncertainty unexplored regions', y=1.02)
plt.tight_layout()
plt.savefig('uncertainty_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Export: one mean-prediction row per grid cell (GP mean = best estimate)
df_gp_dense = pd.DataFrame(X_gw_raw, columns=INPUT_COLS)
for qw in QW_COLS:
    mu, std = gps[qw].predict(X_gw, return_std=True)
    df_gp_dense[qw]          = np.clip(mu, 0, 1)
    df_gp_dense[f'{qw}_std'] = std
df_gp_dense['source'] = 'GP_mean'

# Posterior sample: resample directly on X_gw (20x20 grid)
df_gp_sample = pd.DataFrame(X_gw_raw, columns=INPUT_COLS)
for qw in QW_COLS:
    post_gw = gps[qw].sample_y(X_gw, n_samples=1, random_state=7).ravel()
    df_gp_sample[qw] = np.clip(post_gw, 0, 1)
df_gp_sample['source'] = 'GP_posterior_sample'

df_gp_dense.to_csv('gp_dense_mean_grid.csv',   index=False)
df_gp_sample.to_csv('gp_posterior_sample.csv', index=False)

print(f'GP dense grid (mean):         gp_dense_mean_grid.csv   ({len(df_gp_dense)} rows)')
print(f'GP posterior sample (1 draw): gp_posterior_sample.csv  ({len(df_gp_sample)} rows)')
print(f'\nGP-predicted optimum per campaign target:')
for qw in TARGET_QWS:
    best_r, best_c = np.unravel_index(reward_maps_gp[qw].argmax(), reward_maps_gp[qw].shape)
    camp = {'QW 1':'N1','QW 2':'N2','QW 3':'N3'}[qw]
    print(f'  {qw} ({camp}): T={t_gw[best_r]:.1f}C, R_BAAc={b_gw[best_c]:.3f} '
          f'-> predicted fraction = {reward_maps_gp[qw][best_r,best_c]:.3f}')

## Summary

### What we learned about the campaigns
- **N1** targeted `QW 1` (most quantum-confined, n=1). It explored the widest input space, especially high R_BAAc (up to 2.5) since high BA content suppresses the bulk phase and forces n=1 formation.
- **N2** targeted `QW 2` — pulled the BA ratio down and refined temperature resolution to find the sweet spot between n=1 and n=3.
- **N3** targeted `QW 3` — continued narrowing: R_BAAc down to 0.4–1.1, still exploring temperature. Lower BA content lets the perovskite grow thicker layers (higher n).

**The BO "zoomed in"** progressively — each campaign used the previous one's result as a prior and tightened the search region.

### GP model outputs
| File | Contents |
|------|----------|
| `gp_dense_mean_grid.csv` | 400-point dense grid with GP mean predictions + uncertainty per QW |
| `gp_posterior_sample.csv` | 400-point grid with one posterior draw (includes stochastic variation) |

### How to use in Grid World
In `GridWorld_Simulation.ipynb`, replace the RBF interpolation step with:
```python
df_gp = pd.read_csv('gp_dense_mean_grid.csv')
# Use df_gp[target_qw].values.reshape(20,20) as reward_map directly
```
This gives a GP-derived reward map that correctly models uncertainty in sparse regions.